# Attribution des accessions UniProt aux peptides crosslinkés (XlinkX)

Ce notebook prend l'export XlinkX/Proteome Discoverer (colonnes `Sequence A` / `Sequence B`) et ajoute
pour chaque peptide l'accession UniProt de la protéine d'origine (**Accession A**, **Accession B**).

**Méthode**
1. Les peptides uniques sont envoyés par lots au service **UniProt Peptide Search**
   (`https://peptidesearch.uniprot.org/asyncrest`), restreint à l'organisme choisi
   (*Phaeodactylum tricornutum* par défaut), avec Leu = Ile (indiscernables en MS).
2. Ce service ne renvoie que la *liste* des accessions trouvées (sans dire quel peptide
   correspond à quelle protéine) : on télécharge donc les séquences de ces entrées depuis
   `rest.uniprot.org` puis on refait la correspondance exacte peptide → protéine(s) localement.
3. Pour chaque crosslink, une **accession principale** est choisie parmi les correspondances :
   d'abord celle où la position du peptide correspond à `Leading Protein Position`,
   puis Swiss-Prot (reviewed) avant TrEMBL. La **liste complète** des correspondances est conservée.
4. Ajout du gène, du nom de protéine et du type de lien (intra / inter).

Les résultats intermédiaires sont mis en cache (`cache_uniprot/`) : relancer le notebook ne
refait pas les requêtes.

> Pourquoi pas un BLAST ? Les peptides font en médiane 8 acides aminés : BLAST n'est pas fiable
> sur des séquences aussi courtes et serait très lent pour ~9 000 peptides. Comme les peptides ont été
> identifiés contre une base de données, une correspondance **exacte** est la bonne approche.

In [ ]:
# Dépendances (à décommenter si besoin)
# %pip install pandas requests openpyxl tqdm

import json, re, time, bisect
from pathlib import Path

import pandas as pd
import requests

try:
    from tqdm.auto import tqdm
except ImportError:          # tqdm est optionnel
    def tqdm(x, **k): return x

## 1. Paramètres

In [ ]:
INPUT_CSV = "Crosslink_Data_Anonymized_2.0 - Sheet1.csv"   # chemin vers votre export
OUTPUT_PREFIX = "Crosslink_with_accessions"                  # -> .csv et .xlsx

# Organisme : Phaeodactylum tricornutum (espèce 2850) + souche de référence CCAP 1055/1 (556484).
# Mettre TAX_IDS = [] pour chercher dans tout UniProtKB (déconseillé : un peptide de 8 aa
# correspond à des centaines d'organismes).
TAX_IDS = [2850, 556484]

LEU_EQ_ILE = True        # Leu et Ile considérés comme identiques
SWISSPROT_ONLY = False   # True = seulement les entrées Swiss-Prot (très peu chez P. tricornutum)
BATCH_SIZE = 200         # peptides par job Peptide Search
CACHE_DIR = Path("cache_uniprot")

# Si le service Peptide Search est indisponible, télécharger tout le protéome UniProt
# de l'organisme (TAX_IDS) et faire la recherche exacte localement.
FALLBACK_FULL_PROTEOME = True

PEPSEARCH_URL = "https://peptidesearch.uniprot.org/asyncrest"
UNIPROT_REST = "https://rest.uniprot.org/uniprotkb"
FIELDS = "accession,reviewed,gene_names,protein_name,organism_name,organism_id,sequence"

CACHE_DIR.mkdir(exist_ok=True)
session = requests.Session()
session.headers["User-Agent"] = "crosslink-accession-notebook (python-requests)"

## 2. Lecture des données

In [ ]:
df = pd.read_csv(INPUT_CSV)
df = df[df["Sequence A"].notna() & df["Sequence B"].notna()].copy()

def clean(seq):
    # garde uniquement les acides aminés (au cas où des modifs seraient écrites dans la séquence)
    return re.sub(r"[^A-Z]", "", str(seq).upper())

df["_pepA"] = df["Sequence A"].map(clean)
df["_pepB"] = df["Sequence B"].map(clean)
peptides = sorted(set(df["_pepA"]) | set(df["_pepB"]))

print(f"{len(df)} identifications, {df['Sequence'].nunique()} paires uniques, {len(peptides)} peptides uniques")

## 3. UniProt Peptide Search (par lots)

In [ ]:
ACC_RE = re.compile(r"\b(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-\d+)?\b")

def peptide_search(peps, max_wait=1800):
    # Soumet un job Peptide Search et renvoie l'ensemble des accessions trouvées.
    data = {"peps": ",".join(peps)}
    if TAX_IDS:
        data["taxIds"] = ",".join(map(str, TAX_IDS))
    if LEU_EQ_ILE:
        data["lEQi"] = "on"
    if SWISSPROT_ONLY:
        data["spOnly"] = "on"

    for attempt in range(5):
        r = session.post(PEPSEARCH_URL, data=data, allow_redirects=False, timeout=120)
        if r.status_code not in (429, 500, 502, 503, 504):
            break
        time.sleep(2 ** attempt * 5)
    if r.status_code == 200:            # réponse directe
        return set(ACC_RE.findall(r.text))
    if r.status_code not in (201, 202, 303) or "Location" not in r.headers:
        raise RuntimeError(f"Soumission refusée ({r.status_code}): {r.text[:300]}")

    job_url = requests.compat.urljoin(PEPSEARCH_URL + "/", r.headers["Location"])
    t0 = time.time()
    while time.time() - t0 < max_wait:
        r = session.get(job_url, allow_redirects=False, timeout=120)
        if r.status_code == 200:
            return set(ACC_RE.findall(r.text))
        if r.status_code == 303:        # job en cours (ou redirection vers les résultats)
            nxt = r.headers.get("Location")
            if nxt:
                job_url = requests.compat.urljoin(job_url, nxt)
            time.sleep(int(r.headers.get("Retry-After", 5)))
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(15)
            continue
        raise RuntimeError(f"Erreur job {job_url} ({r.status_code}): {r.text[:300]}")
    raise TimeoutError(f"Job trop long : {job_url}")


pepsearch_cache = CACHE_DIR / "peptide_search_batches.json"
done = json.loads(pepsearch_cache.read_text()) if pepsearch_cache.exists() else {}

batches = [peptides[i:i + BATCH_SIZE] for i in range(0, len(peptides), BATCH_SIZE)]
pepsearch_ok = True
for batch in tqdm(batches, desc="Peptide Search"):
    key = ",".join(batch)
    if key in done:
        continue
    try:
        done[key] = sorted(peptide_search(batch))
    except Exception as e:
        print("Peptide Search a échoué :", e)
        pepsearch_ok = False
        break
    pepsearch_cache.write_text(json.dumps(done))

accessions = sorted({a for accs in done.values() for a in accs})
print(f"{len(accessions)} accessions UniProt trouvées ; Peptide Search complet : {pepsearch_ok}")

## 4. Récupération des séquences et annotations UniProt

In [ ]:
def read_tsv(text):
    from io import StringIO
    return pd.read_csv(StringIO(text), sep="\t", dtype=str).fillna("")

def get_with_retry(url, params, tries=5):
    for attempt in range(tries):
        r = session.get(url, params=params, timeout=300)
        if r.ok:
            return r
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt * 5)
            continue
        r.raise_for_status()
    r.raise_for_status()

def fetch_entries(accs, chunk=100):
    # Annotations + séquences pour une liste d'accessions (isoformes -> entrée canonique).
    base = sorted({a.split("-")[0] for a in accs})
    frames = []
    for i in tqdm(range(0, len(base), chunk), desc="UniProt entries"):
        ids = base[i:i + chunk]
        r = get_with_retry(f"{UNIPROT_REST}/accessions",
                           {"accessions": ",".join(ids), "fields": FIELDS, "format": "tsv"})
        frames.append(read_tsv(r.text))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def fetch_proteome(tax_ids):
    query = " OR ".join(f"taxonomy_id:{t}" for t in tax_ids)
    if SWISSPROT_ONLY:
        query = f"({query}) AND reviewed:true"
    r = get_with_retry(f"{UNIPROT_REST}/stream", {"query": query, "fields": FIELDS, "format": "tsv"})
    return read_tsv(r.text)

entries_cache = CACHE_DIR / "entries.tsv"
use_full = (not pepsearch_ok or not accessions) and FALLBACK_FULL_PROTEOME and TAX_IDS
if use_full:
    print("Repli : téléchargement du protéome UniProt complet pour", TAX_IDS)
    entries = fetch_proteome(TAX_IDS)
    entries_cache = CACHE_DIR / "entries_full_proteome.tsv"
elif entries_cache.exists() and set(read_tsv(entries_cache.read_text())["Entry"]) >= {a.split("-")[0] for a in accessions}:
    entries = read_tsv(entries_cache.read_text())
else:
    entries = fetch_entries(accessions)
entries.to_csv(entries_cache, sep="\t", index=False)

entries = entries.rename(columns={"Entry": "accession", "Reviewed": "reviewed", "Gene Names": "gene",
                                  "Protein names": "protein", "Organism": "organism",
                                  "Organism (ID)": "taxid", "Sequence": "sequence"})
entries["reviewed"] = entries["reviewed"].str.lower().eq("reviewed")
entries = entries.drop_duplicates("accession").reset_index(drop=True)
print(len(entries), "entrées UniProt chargées")

## 5. Correspondance exacte peptide → protéine(s)

In [ ]:
def norm(s):
    return s.replace("I", "L") if LEU_EQ_ILE else s

# Toutes les séquences concaténées (séparateur '#') pour une recherche rapide.
starts, parts, pos = [], [], 0
for s in entries["sequence"]:
    starts.append(pos)
    parts.append(norm(s))
    pos += len(s) + 1
big = "#".join(parts)

def find_matches(pep):
    # Renvoie {index_entrée: [positions de début, 1-based]}.
    hits, p, i = {}, norm(pep), big.find(norm(pep))
    while i != -1:
        k = bisect.bisect_right(starts, i) - 1
        hits.setdefault(k, []).append(i - starts[k] + 1)
        i = big.find(p, i + 1)
    return hits

pep_hits = {pep: find_matches(pep) for pep in tqdm(peptides, desc="Matching")}
n_found = sum(bool(h) for h in pep_hits.values())
print(f"{n_found}/{len(peptides)} peptides retrouvés dans UniProt ({len(peptides) - n_found} sans correspondance)")

## 6. Accession principale + liste complète pour chaque crosslink

Choix de l'accession principale parmi les protéines contenant le peptide :
1. la position correspond à `Leading Protein Position` (début du peptide, ou position du résidu crosslinké) ;
2. Swiss-Prot (reviewed) avant TrEMBL ;
3. ordre alphabétique de l'accession (pour un résultat reproductible).

In [ ]:
E = entries.to_dict("records")

def pick(pep, lead_pos, xl_pos):
    hits = pep_hits.get(pep, {})
    if not hits:
        return None, [], False
    def pos_ok(k):
        if pd.isna(lead_pos):
            return False
        lp, x = int(lead_pos), (int(xl_pos) if not pd.isna(xl_pos) else 1)
        return any(lp in (st, st + x - 1) for st in hits[k])
    ranked = sorted(hits, key=lambda k: (not pos_ok(k), not E[k]["reviewed"], E[k]["accession"]))
    return ranked[0], ranked, pos_ok(ranked[0])

def annotate(row, side):
    k, ranked, ok = pick(row[f"_pep{side}"], row[f"Leading Protein Position {side}"],
                         row[f"Crosslinker Position {side}"])
    if k is None:
        return pd.Series({f"Accession {side}": "", f"Gene {side}": "", f"Protein {side}": "",
                          f"Reviewed {side}": "", f"Position match {side}": False,
                          f"N matches {side}": 0, f"All accessions {side}": "",
                          f"Peptide start {side}": ""})
    e = E[k]
    return pd.Series({
        f"Accession {side}": e["accession"],
        f"Gene {side}": e["gene"].split(" ")[0] if e["gene"] else "",
        f"Protein {side}": e["protein"],
        f"Reviewed {side}": "Swiss-Prot" if e["reviewed"] else "TrEMBL",
        f"Position match {side}": ok,
        f"N matches {side}": len(ranked),
        f"All accessions {side}": ";".join(E[j]["accession"] for j in ranked),
        f"Peptide start {side}": ";".join(map(str, pep_hits[row[f'_pep{side}']][k])),
    })

annot = pd.concat([df.apply(annotate, axis=1, side="A"), df.apply(annotate, axis=1, side="B")], axis=1)

def link_type(r):
    a, b = r["Accession A"], r["Accession B"]
    if not a or not b:
        return "non attribué"
    if a == b:
        return "intra"
    if set(r["All accessions A"].split(";")) & set(r["All accessions B"].split(";")):
        return "ambigu (intra possible)"
    return "inter"

annot["Link type"] = annot.apply(link_type, axis=1)
annot["Organism"] = [E[pick(p, None, None)[0]]["organism"] if pep_hits.get(p) else "" for p in df["_pepA"]]

out = pd.concat([df.drop(columns=["_pepA", "_pepB"]), annot], axis=1)

# Colonnes clés juste après les séquences
key = ["Sequence", "Sequence A", "Accession A", "Gene A", "Protein A",
       "Sequence B", "Accession B", "Gene B", "Protein B", "Link type"]
out = out[key + [c for c in out.columns if c not in key]]
out[key].head(10)

## 7. Résumé et export

In [ ]:
print(out["Link type"].value_counts(), "\n")
for s in "AB":
    print(f"Peptide {s}: position confirmée {out[f'Position match {s}'].mean():.1%} | "
          f"plusieurs correspondances {(out[f'N matches {s}'] > 1).mean():.1%} | "
          f"non trouvé {(out[f'N matches {s}'] == 0).mean():.1%}")

unmatched = sorted(p for p, h in pep_hits.items() if not h)
peptide_table = pd.DataFrame([
    {"Peptide": p, "N matches": len(h),
     "Accessions": ";".join(E[k]["accession"] for k in sorted(h, key=lambda k: (not E[k]["reviewed"], E[k]["accession"]))),
     "Genes": ";".join(dict.fromkeys(E[k]["gene"].split(" ")[0] for k in h if E[k]["gene"]))}
    for p, h in pep_hits.items()])

out.to_csv(f"{OUTPUT_PREFIX}.csv", index=False)
with pd.ExcelWriter(f"{OUTPUT_PREFIX}.xlsx", engine="openpyxl") as xw:
    out.to_excel(xw, sheet_name="Crosslinks", index=False)
    peptide_table.to_excel(xw, sheet_name="Peptides", index=False)
    pd.DataFrame({"Peptide sans correspondance": unmatched}).to_excel(xw, sheet_name="Non trouvés", index=False)

print(f"\nÉcrit : {OUTPUT_PREFIX}.csv et {OUTPUT_PREFIX}.xlsx")